# 调用线上的大模型

## 1. 调用具体模型厂商的API

### 1.1 Deepseek

举例一: 从环境变量中读取DEEPSEEK_API_KEY

In [3]:
import sys
print(sys.executable)

/Users/skk/Developer/lang-chain-learn/.venv/bin/python


In [9]:
import os
from langchain_deepseek import ChatDeepSeek
from dotenv import load_dotenv

load_dotenv(override=True, verbose=True,dotenv_path="conf/.env")

chat_model = ChatDeepSeek(
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL"),
    model_name="deepseek-v4-flash",
)
response = chat_model.invoke("你好")
print(response.content)


你好！很高兴见到你 😊

有什么我可以帮你的吗？无论是聊天、解答问题、写作、翻译，还是其他任何事情，都可以告诉我。


举例二: 依靠默认行为读取 .env 环境变量

In [ ]:
from langchain_deepseek import ChatDeepSeek
from dotenv import load_dotenv
# 从.env文件中加载环境变量# override=True 确保.env文件优先
load_dotenv(override=True, verbose=True,dotenv_path="conf/.env")
# 创建DeepSeek LLM
# 系统会自动从环境变量中读取DEEPSEEK_API_KEY和DEEPSEEK_BASE_URL(可点击进入源码)
# Deepseek也设置了默认值,比如base_url, model_name等,这里可以省略
deepseek_llm = ChatDeepSeek(
model="deepseek-v4-flash",
)
print(deepseek_llm.invoke("请介绍一下你自己"))

## 2.1 兼容用法
一方面，LangChain没有为所有大模型厂商提供专用接口，见Langchain大模型集成列表。如果选用的
平台没有专用接口，可以通过兼容接口调用。
另一方面，专用接口的对接方式五花八门，如腾讯混元的ChatHunyuan需要单独的 APP_ID + SecretId+ SecretKey ，配置繁琐，用户不友好。
结论：大多数API平台都支持OpenAI API接口规范，所以基本都可以通过 ChatOpenAI 集成。
举例1：

In [11]:
from langchain_openai import ChatOpenAI

load_dotenv(override=True, verbose=True,dotenv_path="conf/.env")

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

deepseek_llm2 = ChatOpenAI(
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL,
    model_name="deepseek-v4-flash",
)
print(deepseek_llm2.invoke("请介绍一下你自己(openai调用)"))




content='你好！我是一个通过 **OpenAI API 调用的 AI 助手**，在这个对话里扮演 `assistant` 角色。我不是真人，而是基于语言模型生成文本回复的程序。\n\n我可以帮你做这些事情：\n\n- 回答问题、解释概念\n- 写作、润色、翻译、总结\n- 编程、调试、代码解释\n- 头脑风暴、方案分析、学习辅导\n- 根据上下文进行多轮对话\n\n我也有一些限制：\n\n- 可能会出错，重要信息建议核实\n- 知识可能不是最新，除非启用了联网或工具\n- 不能主动访问你的隐私数据或设备\n- 通常不会跨会话记住你，除非调用方在应用层做了记忆功能\n\n简单说，你可以把我看作一个通过 OpenAI API 提供服务的文本智能助手。你想让我帮你做什么？' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 954, 'prompt_tokens': 38, 'total_tokens': 992, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 781, 'rejected_prediction_tokens': None, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 38}, 'model_provider': 'openai', 'model_name': 'deepseek-flash', 'system_fingerprint': 'aeb56401ca74e127821c4f9126dcb669', 'id': 'ed699a97-e95e-49c8-

## 中转站
这就不展开了。

相关依赖:

In [12]:
pip install langchain-openrouter

/Users/skk/Developer/lang-chain-learn/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


环境变量:

OPENROUTER_API_KEY=<YOUR_API_KEY>
OPENROUTER_API_BASE=https://openrouter.ai/api/v1

# 2. 使用统一接口 init_chat_model
之前使用的方法需要记住每个厂商的初始化方式，现在可以使用统一的接口 init_chat_model。

init_chat_model 是 LangChain 1.x 中推出的用于初始化聊天模型的统一接口。只要是LangChain支持的模型都可以处理，它会根据模型名称自动选择对应的模型类初始化实例。

In [14]:
from langchain.chat_models import init_chat_model

model = init_chat_model(
    # provider:model_name 提供商:模型名称
    model="deepseek:deepseek-v4-flash",
    api_key=DEEPSEEK_API_KEY,
    temperature=0.5, # 温度参数(optional) 表示模型的随机性，0表示确定性，1表示随机性最高
    max_tokens=1024, # 最大token数(optional) 表示模型生成的最大token数，默认1024
    # **kwargs 其他参数，根据模型不同而不同
)
print(model.invoke("请介绍一下你自己(openai调用)"))

content='' additional_kwargs={'refusal': None, 'reasoning_content': '我们需要回答用户中文：“请介绍一下你自己(openai调用)”。用户可能是在问“介绍一下你自己”，括号里“openai调用”可能指通过 OpenAI 调用？我们需要作为助手介绍自己。需要遵守政策：不要声称自己是 OpenAI 开发？实际上我们是 AI 助手，由 OpenAI 训练？ 但系统未给出身份。通常介绍：我是 ChatGPT，一个由 OpenAI 开发的大型语言模型，可以协助回答问题、写作、编程、翻译等。但当前 API？如果通过 OpenAI 调用，可能是 API 助手。需要诚实：我不能知道具体调用细节？ 可以说明我是 AI 助手，基于 OpenAI 模型，通过 API 调用。没有个人经历、情感、意识。知识截止 2024-06。能力、限制、使用方式。用户要求“请介绍一下你自己(openai调用)”。需要可能解释“openai调用”是什么意思？ 可能用户想测试身份。我们应简洁有用。用中文回答。\n\n需要注意：如果是 API，系统可能没有特定身份，但通常可以说：我是由 OpenAI 提供的人工智能助手，运行在 ChatGPT/API 接口上。可以处理文本，支持多轮对话，不能主动记忆（除非会话内），可能不能访问互联网/外部工具，除非已启用。知识截止 2024-06。可以生成代码、文本、分析等。不能执行现实世界动作。如果通过 OpenAI API 调用，我每次请求独立，不保留跨会话记忆，除非开发者传上下文或使用存储。需要强调安全与隐私，不要输入敏感信息。\n\n“openai调用”可能指用户明确说这是 OpenAI 调用。那回答可以包括：你正在通过 OpenAI API 调用我。我的回复由模型根据输入生成，没有意识或情感。我不是人类，不会“记得”你，除非上下文包含之前消息。若开发者设置了 system/developer 提示，我会遵循。我可能产生错误，请核实关键信息。\n\n可以给出格式：身份、能力、限制、调用方式、注意事项。最后问有什么可以帮。要避免过度冗长。Desired oververbose 5. 可以写 3-5 段。\n\n需要小心：OpenAI 模型名称？ 不应声称具体模型版本，因为未提供。可以称“我是 OpenAI

问题3： init_chat_model 和直接使用 ChatTongyi、ChatOpenAI、ChatDeepSeek有什么区别？

回答： init_chat_model 是 LangChain 1.0 的统一接口，优势包括：

1. 统一接口：无需记住每个提供商的不同初始化方式（以一致的方式初始化）

2. 易于切换：简化了智能体系统中模型切换策略（只需修改模型字符串）

3. 简洁明了：更简洁的语法，减少样板代码

4. 自动适配：内部根据模型标识自动选择对应的驱动类(ChatOpenAI、ChatDeepSeek)


问题1：model_provider支持哪些provider？

1. model_provider 表示模型的提供者，支持的providers有： anthropic , anthropic_bedrock,azure_ai 等。

2. 如果 model_provider="openai" ，会自动加载 langchain-openai 的依赖包，底层调用的是ChatOpenAI 类。

3. 如果 model_provider="deepseek" ，会自动加载 langchain-deepseek 的依赖包，底层调用的是 ChatDeepSeek 类。

4. 像阿里的 dashscope 尚未被LangChain官方纳入模型的统一注册体系，暂时不知道"dashscope"的提供者是谁。此时可以将model_provider设置为openai，底层将会用openai的规范处理请求，这就要求我们调用的模型服务是OpenAI Compatible的。

问题2：如果在model参数中没有指明模型提供者，必须在model_provider中指明吗？

1. 可以在model参数中通过前缀指定模型供应商，和模型名称之间用 冒号分割 ，等价于通过model_provider参数指定供应商。
2. 如果两个位置都没有指明供应商，LangChain底层会按照内置规则自动推断。
3. 但是，并非所有的模型都支持自动推断，如model名称 qwen-plus 不支持自动推断，没有指明供应商会报错。

# 总结
以DeepSeek为例，我们有4种方式创建模型：
1. 直接使用 ChatDeepSeek 类
2. 通过 init_chat_model 函数
3. 通过 ChatOpenAI 兼容类
4. 通过 coding plan 指定的方式创建模型